# rl17_grid

> Phase 17 (`specs/multiagent/phase17/`): RL that makes larger teams pay off. The team-size evaluation grid (`teamSizeSuite`) and step 0's models; `RL17Config` (constrained replies, clipped passes per batch, mixed team sizes with liars, pilot v3's value-token loss and per-token clip) with `configHash17` (Phase 12-16 hashes unchanged); the pilot (`PILOT17`, `PILOT17_PLAN`, `pilotChoice`), the main arms and runs (`RL17_ARMS`, `RL17_RUNS`, `RL17_PLAN`, `rl17Config`), run claiming across Colab accounts (`claim`) and the main-run cost projection (`projectHours`).

In [ ]:
#| default_exp rl17_grid

In [ ]:
#| hide
from nbdev.showdoc import *

## Evaluation grid and step 0

In [ ]:
#| export
import hashlib
import json
import os
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np

from explRL.clue_rl import RLConfig, runFields
from explRL.coop_grid import MAIN_SEEDS, RL_PLAN, evalCell

SIZES, BIG_SEEDS = (4, 8, 16, 32), list(range(900, 910))
STEP0_MODELS = dict(warm=None, individual='runs_v2/individual', team='runs_v2/team', credit='runs_v2/credit',
                    **{n: f'runs_p13/{n}' for n in ('credit_s2', 'indpen_s1', 'indpen_s2', 'credturn_s1', 'credturn_s2')},
                    **{n: f'runs_p16/{n}' for n in ('individual_s2', 'individual_s3', 'individual_s4', 'indpen_s3',
                                                    'indpen_s4', 'indpen_s5')})   # name -> run dir under OUT (final/)


def teamSizeSuite(models):
    """Per model: clean boards at N 4 (the 15 main seeds) and N 8, 16, 32 (seeds 900-909); boards with N // 4 scripted
    liars at every N (seeds 900-909); 40 turns, blackboard, notes memory; levels '<model>|n<N>' and '<model>|n<N>liar'."""
    cells = []
    for m in models:
        for N in SIZES:
            cells += [evalCell(m, f'n{N}', 'blackboard', N, sd) for sd in (MAIN_SEEDS if N == 4 else BIG_SEEDS)]
            cells += [evalCell(m, f'n{N}liar', 'blackboard', N, sd, nLiar=N // 4) for sd in BIG_SEEDS]
    return cells

## The configuration and its hash

At the defaults an `RL17Config` is `trainRL`'s algorithm, and `configHash17` leaves the new fields out, so it equals `configHash` of the equal `RLConfig`.

In [ ]:
#| export
@dataclass
class RL17Config(RLConfig):
    guided: object = False      # replies constrained while sampling (vLLM structured outputs): True schema order, 'model' its own
    epochs: int = 1             # update passes per batch
    clip: float = None          # ratio clip of the passes (None: Phase 12's loss, no ratio)
    mixN: tuple = ((4, 1.0),)   # team sizes and their shares of each step's boards
    liarShare: float = 0.0      # share of the boards of each team size with N // 4 scripted liars in the first slots
    lrFixed: float = None       # a fixed learning rate (the per-step controller off)
    valueOnly: bool = False     # loss and KL on value tokens only (pilot v3: keys and punctuation are fixed by the schema)
    tokenClip: bool = False     # PPO's per-token clipped ratio instead of the reply-level mean (pilot v3)


RL17_DEFAULTS = dict(guided=False, epochs=1, clip=None, mixN=((4, 1.0),), liarShare=0.0, lrFixed=None, valueOnly=False,
                     tokenClip=False)


def runFields17(cfg):
    d = runFields(cfg)
    return {k: v for k, v in d.items() if k not in RL17_DEFAULTS or v != RL17_DEFAULTS[k]}


def configHash17(cfg):
    "configHash's rule over runFields17: an RL17Config at the new fields' defaults has its RLConfig's hash."
    d = {k: v for k, v in runFields17(cfg).items() if k not in ('out', 'device', 'ckptEvery', 'microBatch', 'steps', 'gradCkpt')}
    return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:12]

## Pilot, arms, plans

In [ ]:
#| export
FIX17 = dict(valueOnly=True, tokenClip=True, maxLen=3072)   # pilot v3's fix (user-approved 2026-10-19); 8-agent prompts reach ~2,200 tokens
BASE17 = dict(scheme='credit', wrongPenalty=2.5, haltJump=0.15, guided='model', turns=20, **FIX17)   # credit, loophole priced
PILOT17_V1 = dict(frozen='626bead', hash='f4b4bd036996', guided=True, dir='runs_p17pilot', choice='p17a',
                  finding='the schema forced the key order action-first, which the warm start never writes (it puts notes first in '
                          '99.9% of 10,880 replies); the trainer scores the forced tokens unconstrained, so the four-pass recipes '
                          'halted on KL (12.8 at step 2, 23,720 at step 1), p17a undid its first three updates, and invalid replies '
                          'stayed at 1-2%; amended by the user on 2026-10-19: same recipes, the model\'s own key order')
PILOT17_V2 = dict(frozen='8b17f72', hash='52b7cfa582fb', dir='runs_p17pilot2', choice=None,
                  finding='every recipe halted on KL (p17a 0.65 at step 28, p17b 0.64 at step 1, p17c 4.6e6 at step 2); on the Mac, '
                          'replayed prompts showed single tokens the warm start writes with probability ~1 (a schema-forced key, '
                          'an empty value) pushed down 4-16 nats, a route the reply-level clip allows (one token may fall about '
                          'clip x length nats), and 89% of 8-agent prompts cut from the left by maxLen 1536; fixed by FIX17')
RL17_ARMS = dict(r17mix=dict(mixN=((4, 0.5), (8, 0.5)), liarShare=0.25), r17four=dict(mixN=((4, 1.0),), liarShare=0.0))
PILOT17 = dict(p17a=dict(epochs=1, clip=None, kl=0.04, klTarget=0.003),
               p17b=dict(epochs=4, clip=0.2, kl=0.01, klTarget=0.01),
               p17c=dict(epochs=4, clip=0.2, kl=0.0, klTarget=0.0, lrFixed=1e-4))   # p17d: the better of b, c with turns 40
PILOT_STEPS, MAIN_STEPS = 30, 200
RL17_SEEDS = dict(r17mix=8, r17four=5)   # 8: exact sign-flip p down to 1/256, so Holm over 3 can reach 0.012 (5 seeds: 1/32 x 3 > 0.05)
RL17_RUNS = [(a, s) for s in range(1, 9) for a in RL17_ARMS if s <= RL17_SEEDS[a]]   # interleaved
SELECT = dict(maxInvalid=0.05, last=10, window=(21, 30), tie=0.02, maxCut=0)
PILOT17_PLAN = dict(
    version=3, frozen=True, previous=[PILOT17_V1, PILOT17_V2], fix=FIX17, dir='runs_p17pilot3', zip='explRL_rl17pilot3.zip',
    base=dict(RL_PLAN=RL_PLAN['version'], common=BASE17, arm=RL17_ARMS['r17mix']), configs=PILOT17,
    p17d='turns 40 on the better of p17b and p17c by the selection rule applied to those two', steps=PILOT_STEPS, seed=1,
    selection='drop a configuration that halted, cut any training prompt (v3), or whose invalid share over its last 10 steps exceeds 0.05; among the rest '
              'the highest mean training welfare over steps 21-30 (the same board seeds at every step for every '
              'configuration: paired); within 0.02 of the best, the lowest mean seconds per step; a screen, one seed',
    measures='per step: rolloutSeconds, updateSeconds, samples, welfare, invalid, cut, kl (value tokens), klAll, halts', exploratory=True)
RL17_PLAN = dict(
    version=1, frozen=False, recipe=None, arms=RL17_ARMS, seeds=RL17_SEEDS, runs=[list(r) for r in RL17_RUNS], steps=MAIN_STEPS,
    common=BASE17, seedRule='r17mix 8 seeds: with 5, an exact sign-flip test cannot go below 1/32, so Holm over the primary three '
    'could never reach 0.05; r17four 5 seeds (H17-5, a permutation test of 8 against 5, reaches 1/1287)',
    projectedHours=None, userConfirmed=False, control='the Phase 12 warm start (step-0 rows)',
    evaluation='teamSizeSuite: clean N 4 (15 main seeds), N 8/16/32 (900-909); N // 4 liars at every N (900-909); 40 turns',
    hypotheses=dict(H17_1='signFlip(gain_n4[r17mix]) > 0', H17_2='signFlip(gain_n32 - gain_n4 [r17mix]) > 0',
                    H17_3='signFlip(pooledShare_n32[r17mix] - warm) > 0',
                    H17_4='signFlip(honestLoss_n32[warm] - honestLoss_n32[r17mix]) > 0',
                    H17_5='permTest(gain_n32[r17mix], gain_n32[r17four]) > 0'),
    families=dict(primary=['H17_1', 'H17_2', 'H17_3'], secondary=['H17_4', 'H17_5']), alpha=0.05,
    intervals='hierarchical bootstrap (runs, then boards), descriptive', haiku='references only (full memory)',
    cutRule='if a 32-agent evaluation cell runs out of memory: N 32 on seeds 900-905 only, for every model', exploratory=True)


def rl17Config(arm, seed, model, out, recipe, steps=MAIN_STEPS):
    """An RL17 run: RL_PLAN v2 + BASE17 + the recipe + the arm; trainSeed = seed - 1; a checkpoint every step (a step takes
    many minutes and a Colab session ends after about 24 hours; the interval is not part of the configuration hash)."""
    return RL17Config(model, out, **{**RL_PLAN['rl'], **BASE17, **recipe, **RL17_ARMS[arm], 'steps': steps,
                                     'trainSeed': seed - 1, 'ckptEvery': 1})


def pilotConfig(name, model, out, recipe=None):
    "A pilot run (r17mix's mix, seed 1, 30 steps); p17d needs the recipe it extends (turns 40)."
    rec = dict(recipe, turns=40) if name == 'p17d' else PILOT17[name]
    return rl17Config('r17mix', 1, model, out, rec, PILOT_STEPS)


def pilotChoice(metrics, halted, names=None, rule=SELECT):
    "The selection rule: metrics {name: [step dicts]}, halted {names}; -> (choice, table {name: dict})."
    a, b = rule['window']
    table = {}
    for n in names or sorted(metrics):
        m = metrics.get(n, [])                         # a run that logged no step does not pass
        inv = float(np.mean([x['invalid'] for x in m[-rule['last']:]])) if m else 1.0
        w = [x['welfare'] for x in m if a <= x['step'] <= b]
        sec = float(np.mean([x['rolloutSeconds'] + x['updateSeconds'] for x in m])) if m else float('inf')
        cut = sum(x.get('cut', 0) for x in m)
        table[n] = dict(halted=n in halted, invalid=inv, welfare=float(np.mean(w)) if w else float('nan'), seconds=sec, cut=cut,
                        ok=n not in halted and inv <= rule['maxInvalid'] and len(w) == b - a + 1 and cut <= rule.get('maxCut', 0))
    ok = [n for n, t in table.items() if t['ok']]
    if not ok:
        return None, table
    best = max(table[n]['welfare'] for n in ok)
    return min((n for n in ok if table[n]['welfare'] >= best - rule['tie']), key=lambda n: table[n]['seconds']), table


def chosenRecipe(choice, p17dBase=None):
    "RL17_PLAN's recipe from the pilot's choice: PILOT17[choice], or p17d's base with 40 turns."
    if choice == 'p17d':
        return dict(PILOT17[p17dBase], turns=40)
    return dict(PILOT17[choice])


def projectHours(stepSeconds, steps=MAIN_STEPS, seeds=RL17_SEEDS, fourRatio=2 / 3):
    "Main-run A100 hours from the chosen pilot's mean seconds per step (r17mix's mix); r17four plays about 2/3 the replies."
    mix = stepSeconds * steps / 3600
    return dict(r17mix=mix, r17four=mix * fourRatio, total=seeds['r17mix'] * mix + seeds['r17four'] * mix * fourRatio)

## Claiming runs across Colab accounts

In [ ]:
#| export
def claim(runDir, host, takeover=()):
    """Claim runDir for `host` (claim.json): free -> claimed; ours -> True; another host's -> False unless the run's name
    is in `takeover`. A folder with progress but no claim (a run started by a cell without claiming) counts as held by
    another host. The write is atomic (temporary file, then rename) and read back."""
    if not host:
        raise ValueError('set HOST in the configuration cell (a different name on each Colab account)')
    d = Path(runDir)
    d.mkdir(parents=True, exist_ok=True)
    p = d / 'claim.json'
    if p.exists():
        cur = json.loads(p.read_text())['host']
        if cur == host:
            return True
        if d.name not in takeover:
            return False
    elif (d / 'metrics.jsonl').exists() or any(d.glob('ckpt-*')) or (d / 'final').exists():
        if d.name not in takeover:
            return False
    tmp = d / f'.claim-{host}-{os.getpid()}.json'
    tmp.write_text(json.dumps(dict(host=host, time=time.time())) + '\n')
    os.replace(tmp, p)
    return json.loads(p.read_text())['host'] == host

## Tests

In [ ]:
import json, tempfile
from dataclasses import asdict
from pathlib import Path
from explRL.rl17_grid import *
from explRL.clue_rl import RLConfig, configHash
from explRL.coop_grid import RL_PLAN
from explRL.rl16_grid import rl16Config

# AC-2: the grid
cells = teamSizeSuite(['warm', 'x'])
per = 15 + 10 + 10 + 10 + 4 * 10
assert len(cells) == 2 * per
lv = lambda c: c['level']
assert {lv(c) for c in cells if c['level'].startswith('warm|')} == {f'warm|n{N}{s}' for N in (4, 8, 16, 32) for s in ('', 'liar')}
assert all(c['cfg'].get('nLiar', 0) == (c['cfg']['N'] // 4 if lv(c).endswith('liar') else 0) for c in cells)
assert all(c['cfg']['turns'] == 40 and c['cfg']['memory'] == 'notes' and c['cfg']['channel'] == 'blackboard' for c in cells)
assert len(STEP0_MODELS) == 15 and STEP0_MODELS['warm'] is None
# the hash: defaults equal RLConfig's; Phase 12-16 hashes unchanged; any new field changes it
m = '/content/drive/MyDrive/explRL_rl/warm_model'
base = RLConfig(m, 'o', **{**RL_PLAN['rl'], 'scheme': 'individual', 'trainSeed': 0})
assert configHash17(RL17Config(**asdict(base))) == configHash(base) == '542eb4f99c29'
c16 = rl16Config('indpen', 3, m, 'o')
assert configHash17(RL17Config(**asdict(c16))) == configHash(c16)
for k, v in dict(guided=True, epochs=4, clip=0.2, mixN=((4, 0.5), (8, 0.5)), liarShare=0.25, lrFixed=1e-4, valueOnly=True, tokenClip=True).items():
    assert configHash17(RL17Config(**{**asdict(base), k: v})) != configHash(base), k
# runs, pilot and main configurations
assert len(RL17_RUNS) == 13 and [a for a, _ in RL17_RUNS[:4]] == ['r17mix', 'r17four'] * 2 and RL17_RUNS[-3:] == [('r17mix', 6), ('r17mix', 7), ('r17mix', 8)]
c = rl17Config('r17mix', 3, m, 'o', PILOT17['p17b'])
assert (c.trainSeed, c.steps, c.epochs, c.clip, c.guided, c.turns, c.scheme, c.wrongPenalty) == (2, 200, 4, 0.2, 'model', 20, 'credit', 2.5)
assert PILOT17_PLAN['version'] == 3 and PILOT17_PLAN['dir'] not in (PILOT17_V1['dir'], PILOT17_V2['dir']) and PILOT17_V1['guided'] is True
assert (c.valueOnly, c.tokenClip, c.maxLen) == (True, True, 3072)              # pilot v3's fix in every Phase 17 run
v1 = RL17Config(**{**asdict(c), 'guided': True})
v2 = RL17Config(**{**asdict(c), 'valueOnly': False, 'tokenClip': False, 'maxLen': 1536})
assert len({configHash17(v1), configHash17(v2), configHash17(c)}) == 3            # no run resumes an earlier pilot's checkpoint
assert c.mixN == ((4, 0.5), (8, 0.5)) and c.liarShare == 0.25
assert rl17Config('r17four', 1, m, 'o', PILOT17['p17a']).mixN == ((4, 1.0),)
p = pilotConfig('p17d', m, 'o', PILOT17['p17c'])
assert (p.turns, p.steps, p.trainSeed, p.lrFixed, p.epochs, p.ckptEvery) == (40, 30, 0, 1e-4, 4, 1)
assert configHash17(c) == configHash17(RL17Config(**{**asdict(c), 'ckptEvery': 5}))       # the interval is outside the hash
for k in ('p17a', 'p17b', 'p17c'):
    assert pilotConfig(k, m, 'o').steps == 30
json.dumps(PILOT17_PLAN); json.dumps(RL17_PLAN)

In [ ]:
# the selection rule on hand-made pilot metrics
def run(w, inv=0.0, sec=100.0, n=30):
    return [dict(step=s, welfare=w, invalid=inv, rolloutSeconds=sec / 2, updateSeconds=sec / 2) for s in range(1, n + 1)]
ch, tab = pilotChoice(dict(p17a=run(0.30), p17b=run(0.31, sec=300), p17c=run(0.40, inv=0.2), p17d=run(0.5)), halted={'p17d'})
assert ch == 'p17a' and not tab['p17c']['ok'] and not tab['p17d']['ok']           # b within 0.02 of a but slower
assert pilotChoice(dict(p17a=run(0.2), p17b=run(0.3)), set())[0] == 'p17b'
assert pilotChoice(dict(p17a=run(0.2, n=12)), set())[0] is None                     # stopped before the window
assert pilotChoice(dict(p17b=run(0.3)), set(), names=['p17b', 'p17c'])[0] == 'p17b'   # p17c logged nothing
assert chosenRecipe('p17d', 'p17b') == dict(PILOT17['p17b'], turns=40) and chosenRecipe('p17a') == PILOT17['p17a']
assert pilotChoice(dict(p17b=run(0.3), p17c=run(0.35)), set(), names=['p17b', 'p17c'])[0] == 'p17c'
cutRun = run(0.5)
cutRun[3]['cut'] = 2
assert pilotChoice(dict(p17a=run(0.3), p17b=cutRun), set())[0] == 'p17a'            # v3: a run that cut a prompt fails
h = projectHours(1360)
assert abs(h['r17mix'] - 1360 * 200 / 3600) < 1e-9 and abs(h['total'] - (8 + 5 * 2 / 3) * h['r17mix']) < 1e-9

In [ ]:
# AC-4: claiming with two simulated hosts on one folder
with tempfile.TemporaryDirectory() as d:
    r = Path(d) / 'r17mix_s1'
    assert claim(r, 'a') and claim(r, 'a') and not claim(r, 'b')
    assert claim(r, 'b', takeover=['r17mix_s1']) and not claim(r, 'a')                 # explicit takeover only
    e = Path(d) / 'eval_p17'
    assert claim(e, 'b') and not claim(e, 'a')                                          # one host evaluates
    u = Path(d) / 'p17a'
    (u / 'ckpt-0003').mkdir(parents=True)                                              # progress without a claim
    assert not claim(u, 'a') and claim(u, 'a', takeover=['p17a']) and not claim(u, 'b')
    try:
        claim(r, '')
        raise AssertionError
    except ValueError:
        pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()